In [20]:
import os
import click
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from utils.datasets import load_data, HandwrittenData, OCRcollateFn
from utils.preprocessing import process_img
from core.tokenizer import Tokenizer
from core.crnn import HandwrittenCRNN
from torch.utils.data import DataLoader

In [2]:
data_dict_parents = ["data", "train", "PERFECT_CUT_a_z_1_9"]
data_dict_parents = os.path.join(*data_dict_parents)
data_dict_file = "0annotation.json"
data_dict_path = os.path.join(data_dict_parents, data_dict_file)

datadict = load_data(data_dict_path)

tokens_file = os.path.join("include", "tokens.json")
tokenizer = Tokenizer(src=tokens_file)

collate_fn = OCRcollateFn(tokenizer)

dataset = HandwrittenData(
    datadict=datadict, data_path=data_dict_parents, transform=process_img
)

In [18]:
data_loader_train = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

model = HandwrittenCRNN(num_classes=len(tokenizer._encode)).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)

Entrenando en: cpu


In [16]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, targets, target_lengths) in enumerate(dataloader):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()

        logits = model(images)
        log_probs = F.log_softmax(logits, dim=2)
        log_probs = log_probs.permute(1, 0, 2)

        batch_size = images.size(0)
        tiempo_prediccion = log_probs.size(0) # 31
        
        input_lengths = torch.full(
            size=(batch_size,), 
            fill_value=tiempo_prediccion, 
            dtype=torch.long
        ).to(device)

        loss = criterion(log_probs, targets, input_lengths, target_lengths)

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        running_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Lote [{batch_idx}/{len(dataloader)}] - Loss: {loss.item():.4f}")

    return running_loss / len(dataloader)

In [21]:
num_epochs = 30

print("¡Iniciando el entrenamiento!")
for epoch in range(num_epochs):
    loss_promedio = train_one_epoch(model, data_loader_train, criterion, optimizer, device)
    print(f"=== Época {epoch+1}/{num_epochs} completada | Loss Promedio: {loss_promedio:.4f} ===")

¡Iniciando el entrenamiento!
Lote [0/96] - Loss: 36.2500
Lote [50/96] - Loss: 3.7154
=== Época 1/30 completada | Loss Promedio: 4.3192 ===
Lote [0/96] - Loss: 3.3096
Lote [50/96] - Loss: 3.2072
=== Época 2/30 completada | Loss Promedio: 3.1339 ===
Lote [0/96] - Loss: 3.0522
Lote [50/96] - Loss: 2.9650
=== Época 3/30 completada | Loss Promedio: 3.0046 ===
Lote [0/96] - Loss: 2.7966
Lote [50/96] - Loss: 2.8182
=== Época 4/30 completada | Loss Promedio: 2.8490 ===
Lote [0/96] - Loss: 2.7587
Lote [50/96] - Loss: 2.5222


KeyboardInterrupt: 